# <font color="#418FDE" size="6.5" uppercase>**Kernel Approximation**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Approximate kernel feature maps using scikit-learn transformers. 
- Apply random projections to reduce dimensionality while preserving distances approximately. 
- Evaluate approximation quality and computational tradeoffs. 


## **1. Kernel Feature Maps**

### **1.1. Nyström Kernel Approximation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_01_01.jpg?v=1783990263" width="250">



>* Approximate kernels using selected landmark examples
>* Enable linear models to capture nonlinear patterns

>* Use landmarks to approximate kernel similarities
>* More components improve accuracy but cost more

>* Choose kernels and components carefully
>* Balance accuracy, speed, and memory



In [ ]:
#@title Python Code - Nyström Kernel Approximation

# This example approximates nonlinear kernel features.
# Nystroem creates explicit features from landmark samples.
# A linear classifier then learns curved class boundaries.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_circles
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create a small nonlinear classification dataset.
features, labels = make_circles(
    n_samples=600,
    factor=0.35,
    noise=0.08,
    random_state=42,
)

# Check the dataset shape before modeling.
if features.shape != (600, 2):
    raise ValueError("Expected 600 rows and 2 features.")

# Split data before fitting any preprocessing steps.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=42,
)

# Build a plain linear baseline model.
linear_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)

# Build a Nystroem approximation plus linear model.
nystroem_model = make_pipeline(
    StandardScaler(),
    Nystroem(kernel="rbf", gamma=4.0, n_components=80, random_state=42),
    LogisticRegression(max_iter=1000, random_state=42),
)

# Fit both models on the same training data.
linear_model.fit(X_train, y_train)
nystroem_model.fit(X_train, y_train)

# Compare test accuracy for the two approaches.
linear_accuracy = accuracy_score(y_test, linear_model.predict(X_test))
nystroem_accuracy = accuracy_score(y_test, nystroem_model.predict(X_test))

# Transform a few rows to show the new feature size.
transformed_sample = nystroem_model.named_steps["nystroem"].transform(
    nystroem_model.named_steps["standardscaler"].transform(X_train[:5])
)

print("scikit-learn version:", sklearn.__version__)
print("Original feature count:", X_train.shape[1])
print("Nystroem feature count:", transformed_sample.shape[1])
print("Linear test accuracy:", round(linear_accuracy, 3))
print("Nystroem plus linear test accuracy:", round(nystroem_accuracy, 3))

# Plot the data to reveal the nonlinear pattern.
fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=labels,
    cmap="coolwarm",
    s=25,
    edgecolor="k",
)

# Label the single teaching plot clearly.
ax.set_title("Concentric circles need nonlinear features")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend(*scatter.legend_elements(), title="Class")
plt.show()



### **1.2. Random Fourier Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_01_02.jpg?v=1783990265" width="250">



>* Approximate kernels with random sinusoidal features
>* Train scalable linear models on transformed data

>* Scales kernel learning to large datasets
>* More features improve accuracy but cost more

>* Random features approximate kernels with seed variation
>* Scale features and tune components carefully



In [ ]:
#@title Python Code - Random Fourier Features

# This example approximates an RBF kernel map.
# Random Fourier features create finite nonlinear features.
# More components usually improve the kernel approximation.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.kernel_approximation import RBFSampler
from sklearn.metrics.pairwise import rbf_kernel

# Create a small deterministic dataset for distance-based kernels.
features, labels = make_blobs(
    n_samples=120,
    centers=3,
    n_features=2,
    cluster_std=1.2,
    random_state=42,
)

# Validate the shape before comparing kernel matrices.
if features.shape != (120, 2):
    raise ValueError("Expected 120 rows and 2 features.")

# Compute the exact RBF kernel for comparison.
gamma_value = 0.5
exact_kernel = rbf_kernel(features, gamma=gamma_value)

# Try several random feature dimensions.
component_counts = [10, 30, 100, 300]
mean_errors = []

for count in component_counts:
    sampler = RBFSampler(
        gamma=gamma_value,
        n_components=count,
        random_state=42,
    )
    transformed_features = sampler.fit_transform(features)
    approximate_kernel = transformed_features @ transformed_features.T
    mean_error = np.mean(np.abs(exact_kernel - approximate_kernel))
    mean_errors.append(mean_error)

# Print concise results that connect components to approximation quality.
print("scikit-learn version:", sklearn.__version__)
print("Original feature shape:", features.shape)
print("Random Fourier feature errors:")

for count, error in zip(component_counts, mean_errors):
    print(count, "components -> mean absolute error", round(error, 4))

# Plot how approximation error changes with more random features.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(component_counts, mean_errors, marker="o")
ax.set_title("Random Fourier Features Approximate an RBF Kernel")
ax.set_xlabel("Number of random Fourier components")
ax.set_ylabel("Mean absolute kernel error")
ax.grid(True, alpha=0.3)
plt.show()



### **1.3. Chi Squared Maps**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_01_03.jpg?v=1783990267" width="250">



>* Best for nonnegative histogram-like data
>* Approximates relative-bin similarity for linear models

>* Expand histogram features to approximate chi squared kernels
>* Train scalable linear models on transformed features

>* Balance accuracy, features, memory, and speed
>* Use with nonnegative histogram-like data



In [ ]:
#@title Python Code - Chi Squared Maps

# This example transforms nonnegative histogram features.
# Additive chi squared maps approximate nonlinear similarity.
# Linear learning improves after the feature expansion.

import numpy as np
import sklearn
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

# Small histogram-like rows represent counts in four bins.
X = np.array([
    [9, 1, 0, 0], [8, 2, 0, 0], [7, 1, 1, 0], [6, 2, 1, 0],
    [0, 0, 1, 9], [0, 0, 2, 8], [0, 1, 1, 7], [1, 0, 2, 6],
    [5, 5, 0, 0], [4, 6, 0, 0], [0, 0, 5, 5], [0, 0, 4, 6],
], dtype=float)

y = np.array([0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1])

# Chi squared maps require nonnegative histogram values.
if np.any(X < 0):
    raise ValueError("Histogram features must be nonnegative.")

# Split data before fitting the transformer and classifier.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)

# A plain linear model sees only the original histogram bins.
linear_model = LogisticRegression(random_state=42, max_iter=200)
linear_model.fit(X_train, y_train)
linear_accuracy = accuracy_score(y_test, linear_model.predict(X_test))

# The sampler expands each bin into chi squared map features.
chi2_pipeline = make_pipeline(
    AdditiveChi2Sampler(sample_steps=2),
    LogisticRegression(random_state=42, max_iter=200),
)

chi2_pipeline.fit(X_train, y_train)
chi2_accuracy = accuracy_score(y_test, chi2_pipeline.predict(X_test))

# Transform once to show how the feature dimension changes.
chi2_sampler = AdditiveChi2Sampler(sample_steps=2)
X_train_mapped = chi2_sampler.fit_transform(X_train)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Original feature count: {X_train.shape[1]}")
print(f"Mapped feature count: {X_train_mapped.shape[1]}")
print(f"Linear accuracy on raw histograms: {linear_accuracy:.2f}")
print(f"Linear accuracy after chi squared map: {chi2_accuracy:.2f}")



## **2. Polynomial Sketches**

### **2.1. Skewed Chi Squared**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_02_01.jpg?v=1783990274" width="250">



>* Random sketches compress polynomial feature interactions.
>* Skewed errors can cause large deviations.

>* Sketches compress costly customer interaction combinations
>* Dominant rare behaviors can skew distance estimates

>* More features reduce sketch variability
>* Data structure affects approximation reliability



In [ ]:
#@title Python Code - Skewed Chi Squared

# This example sketches polynomial interactions with random projections.
# It compares balanced and heavy-tailed feature data.
# The plot shows skewed approximation errors clearly.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.random_projection import GaussianRandomProjection

# A fixed generator makes the random data reproducible.
rng = np.random.default_rng(42)

# Balanced data has many similarly sized feature values.
balanced_data = rng.normal(loc=0.0, scale=1.0, size=(120, 8))

# Heavy-tailed data has occasional unusually large feature values.
heavy_tailed_data = rng.standard_t(df=2.0, size=(120, 8))
heavy_tailed_data = np.clip(heavy_tailed_data, -8.0, 8.0)

# Squaring features mimics simple polynomial interaction strength.
balanced_poly = balanced_data * balanced_data
heavy_poly = heavy_tailed_data * heavy_tailed_data

# Validate that both examples have the same teaching shape.
if balanced_poly.shape != heavy_poly.shape:
    raise ValueError("Both datasets must have matching shapes.")

# Random projection compresses the polynomial-like representation.
sketcher = GaussianRandomProjection(n_components=3, random_state=42)

# Fit only on balanced data to define one shared projection.
sketcher.fit(balanced_poly)
balanced_sketch = sketcher.transform(balanced_poly)
heavy_sketch = sketcher.transform(heavy_poly)

# Compare true squared lengths with sketched squared lengths.
balanced_true = np.sum(balanced_poly * balanced_poly, axis=1)
balanced_estimate = np.sum(balanced_sketch * balanced_sketch, axis=1)
heavy_true = np.sum(heavy_poly * heavy_poly, axis=1)
heavy_estimate = np.sum(heavy_sketch * heavy_sketch, axis=1)

# Relative error shows how much distance preservation changes.
balanced_error = (balanced_estimate - balanced_true) / balanced_true
heavy_error = (heavy_estimate - heavy_true) / heavy_true

# Print concise summaries of the two error distributions.
print("scikit-learn version: sklearn random projection demo")
print("Balanced median error:", round(float(np.median(balanced_error)), 3))
print("Heavy-tailed median error:", round(float(np.median(heavy_error)), 3))
print("Balanced 95th percentile:", round(float(np.percentile(balanced_error, 95)), 3))
print("Heavy-tailed 95th percentile:", round(float(np.percentile(heavy_error, 95)), 3))

# One histogram highlights the skewed long-tail behavior.
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(balanced_error, bins=25, alpha=0.65, label="balanced data")
ax.hist(heavy_error, bins=25, alpha=0.65, label="heavy-tailed data")
ax.set_title("Random projection error after polynomial-style squaring")
ax.set_xlabel("relative squared-length error")
ax.set_ylabel("number of samples")
ax.legend()
plt.show()



### **2.2. TensorSketch Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_02_02.jpg?v=1783990278" width="250">



>* Compress polynomial interactions into compact sketches
>* Preserve kernel similarities for linear models

>* Sketch interactions without full tensor products
>* Preserve geometry for scalable polynomial learning

>* Sketch size balances accuracy and efficiency
>* Approximates nonlinear patterns when features explode



In [ ]:
#@title Python Code - TensorSketch Features

# TensorSketch approximates polynomial kernel similarities.
# Random sketches trade accuracy for compact dimensions.
# The plot shows approximation improving with size.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.kernel_approximation import PolynomialCountSketch
import sklearn

# Small fixed data keeps the demonstration fast and repeatable.
rng = np.random.default_rng(42)
X = rng.normal(size=(80, 6))

# Validate the shape before comparing pairwise similarities.
if X.shape != (80, 6):
    raise ValueError("Expected 80 samples with 6 features.")

# This is the exact degree-two polynomial kernel matrix.
degree = 2
exact_kernel = (X @ X.T) ** degree
exact_norm = np.linalg.norm(exact_kernel)

# Larger sketches usually preserve polynomial similarities better.
sketch_sizes = [20, 50, 100, 200, 400]
relative_errors = []

for sketch_size in sketch_sizes:
    sketcher = PolynomialCountSketch(
        degree=degree, n_components=sketch_size, random_state=42
    )
    X_sketch = sketcher.fit_transform(X)
    approximate_kernel = X_sketch @ X_sketch.T
    error = np.linalg.norm(approximate_kernel - exact_kernel) / exact_norm
    relative_errors.append(error)

# Print a compact summary for beginners.
print("scikit-learn version:", sklearn.__version__)
print("Original features:", X.shape[1])
print("Exact degree-2 interactions would be much larger.")
print("Best relative kernel error:", round(min(relative_errors), 3))

# Plot approximation error against sketch dimension.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sketch_sizes, relative_errors, marker="o")
ax.set_title("TensorSketch approximation improves with more features")
ax.set_xlabel("TensorSketch output dimensions")
ax.set_ylabel("Relative kernel approximation error")
ax.grid(True, alpha=0.3)
plt.show()



### **2.3. Approximation Limits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_02_03.jpg?v=1783990276" width="250">



>* Sketches compress feature interactions efficiently
>* Too much compression distorts similarity

>* Higher degrees need larger, careful sketches
>* Sparse data sketches better than dense data

>* Smaller sketches trade accuracy for speed
>* Validate dimensions against task needs



## **3. Random Projection Tradeoffs**

### **3.1. Distance Preservation Intuition**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_03_01.jpg?v=1783990271" width="250">



>* Keep nearby points close after projection
>* Preserve useful structure for learning algorithms

>* Compress data while controlling distortion
>* Preserve distances for downstream learning

>* Projection size controls distance accuracy
>* Balance speed gains with model quality



### **3.2. Gaussian Random Projection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_03_02.jpg?v=1783990269" width="250">



>* Reduce dimensions with Gaussian random matrices
>* Preserve distances in very large feature spaces

>* Fewer dimensions speed learning but risk distortion
>* Evaluate projection size using task performance

>* Dense projections can slow sparse data
>* Balance accuracy, speed, memory, and performance



In [ ]:
#@title Python Code - Gaussian Random Projection

# This example compares Gaussian projection sizes.
# Random projection trades speed for distance accuracy.
# The plot shows distortion decreasing with dimensions.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.random_projection import GaussianRandomProjection
from sklearn.metrics import pairwise_distances
import sklearn

# A fixed generator makes the synthetic data reproducible.
rng = np.random.default_rng(42)
original_data = rng.normal(size=(300, 80))

# Validate the data shape before measuring distances.
if original_data.shape != (300, 80):
    raise ValueError("Expected 300 samples with 80 features.")

# Original pairwise distances are the reference geometry.
original_distances = pairwise_distances(original_data)
upper_triangle = np.triu_indices(original_data.shape[0], k=1)
reference_distances = original_distances[upper_triangle]

# Test several projected dimensions as the tradeoff knob.
projected_dimensions = [5, 10, 20, 40, 60]
mean_relative_errors = []

for dimension in projected_dimensions:
    projector = GaussianRandomProjection(
        n_components=dimension,
        random_state=42,
    )
    projected_data = projector.fit_transform(original_data)
    projected_distances = pairwise_distances(projected_data)[upper_triangle]
    relative_errors = np.abs(projected_distances - reference_distances)
    relative_errors = relative_errors / reference_distances
    mean_relative_errors.append(float(np.mean(relative_errors)))

# Print concise context for the plotted tradeoff.
print("scikit-learn version:", sklearn.__version__)
print("Original shape:", original_data.shape)
print("Best mean distance error:", round(min(mean_relative_errors), 3))

# Plot one curve showing approximation quality versus size.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(projected_dimensions, mean_relative_errors, marker="o")
ax.set_title("Gaussian Random Projection Tradeoff")
ax.set_xlabel("Projected dimensions")
ax.set_ylabel("Mean relative distance error")
ax.grid(True, alpha=0.3)
plt.show()



### **3.3. Sparse Projection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_03_03.jpg?v=1783990272" width="250">



>* Mostly zero matrices reduce projection computation
>* Randomness preserves distances for sparse data

>* Faster projections with lower memory use
>* Scales well for sparse, high-dimensional data

>* Sparse projections trade precision for efficiency
>* Evaluate neighbors, performance, speed, and memory



In [ ]:
#@title Python Code - Sparse Projection

# Sparse projection trades accuracy for faster distance estimates.
# This example compares several projected feature sizes.
# The plot shows distortion shrinking with more dimensions.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.random_projection import SparseRandomProjection
from sklearn.metrics import pairwise_distances

# Generate high-dimensional data with a fixed random seed.
features, labels = make_classification(
    n_samples=600,
    n_features=1000,
    n_informative=40,
    random_state=42,
)

# Validate the shape before measuring projection quality.
if features.shape != (600, 1000):
    raise ValueError("Expected 600 rows and 1000 features.")

# Measure original distances on a small deterministic subset.
subset = features[:120]
original_distances = pairwise_distances(subset)
mask = np.triu(np.ones(original_distances.shape, dtype=bool), k=1)
original_pairs = original_distances[mask]

# Compare sparse projections with different target dimensions.
target_dimensions = [25, 50, 100, 200, 400]
mean_relative_errors = []

for target_dimension in target_dimensions:
    projector = SparseRandomProjection(
        n_components=target_dimension,
        random_state=42,
    )
    projected = projector.fit_transform(subset)
    projected_distances = pairwise_distances(projected)
    projected_pairs = projected_distances[mask]
    relative_error = np.abs(projected_pairs - original_pairs) / original_pairs
    mean_relative_errors.append(float(np.mean(relative_error)))

# Print concise context for the plotted tradeoff.
print("scikit-learn version:", sklearn.__version__)
print("Original features:", features.shape[1])
print("Best mean relative distance error:", round(min(mean_relative_errors), 3))

# Plot approximation error against projected dimensionality.
plt.figure(figsize=(7, 4))
plt.plot(target_dimensions, mean_relative_errors, marker="o")
plt.title("Sparse Projection Tradeoff")
plt.xlabel("Projected dimensions")
plt.ylabel("Mean relative distance error")
plt.grid(True, alpha=0.3)
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Kernel Approximation**</font>


In this lecture, you learned to:
- Approximate kernel feature maps using scikit-learn transformers. 
- Apply random projections to reduce dimensionality while preserving distances approximately. 
- Evaluate approximation quality and computational tradeoffs. 

In the next Module (Module 9), we will go over 'Linear Regression'